In [ ]:
import pandas as pd
import sys
import os
import json

# Add the project root (one level above notebooks/) to sys.path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)
# print(project_root)

import variant_classification as varclass
from importlib import reload

# Reload the module after making changes
reload(varclass)

variant_dict = {
    "positive": pd.read_csv("/mnt/d/phd/scripts/14_gnomAD_conservation_proj/data/output/combined_joint_variants_pos_win5_extended.tsv", sep="\t"),
    "negative": pd.read_csv("/mnt/d/phd/scripts/14_gnomAD_conservation_proj/data/output/combined_joint_variants_neg_win5_extended.tsv", sep="\t")
}

pr_regions, pr_overlap, df_matched, df_unmatched = varclass.compute_variant_region_overlap_full(
    json_file="/mnt/d/phd/scripts/14_gnomAD_conservation_proj/data/processed/genomic_coordinates_info_merged_win5_extended.json",
    variant_dfs=variant_dict
)

print(len(df_matched), "matched")
print(len(df_unmatched), "unmatched")



### What are the 1084 unmatched ones


In [ ]:
#### NOW BEGINS THE CLASSIFICATION

In [ ]:
# --- Imports and path setup ---------------------------------------------------
import os
import sys
import json
import pandas as pd
from Bio.Seq import Seq

# Add project root (one level above notebooks/) to sys.path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# --- Helper functions ---------------------------------------------------------

def find_interval(intervals, chrom, pos):
    for iv in intervals:
        if iv["chrom"] == chrom and iv["start"] <= pos <= iv["end"]:
            return iv
    return None


def genomic_to_merged_index(pos, iv):
    if iv["strand"] == "+":
        return iv["merged_start"] + (pos - iv["start"])
    else:
        return iv["merged_start"] + (iv["end"] - pos)


def normalize_alleles(ref, alt, strand):
    if strand == "+":
        return ref, alt
    return (
        str(Seq(ref).reverse_complement()),
        str(Seq(alt).reverse_complement()),
    )


def apply_variant(dna, intervals, chrom, pos, ref, alt, strand):
    iv = find_interval(intervals, chrom, pos)
    if iv is None:
        raise ValueError(f"Variant {chrom}:{pos} not covered by intervals")
    ref_old = ref
    ref, alt = normalize_alleles(ref, alt, iv["strand"])
    i = genomic_to_merged_index(pos, iv)
    if strand == "-":
        derived_ref = dna[i - len(ref) + 1 :i + 1]
        # derived_ref = dna[:i + len(ref)]
    else:
        derived_ref = dna[i:i + len(ref)]
    if derived_ref != ref:
        if len(derived_ref) != len(ref):
            # print("Length mismatch:")
            # print(f"REF length: {len(ref)}, found length: {len(derived_ref)}")
            # print("derived ref not ref")
            if derived_ref in ref:
                # print("Derived REF is contained in REF, thats good enough it just goes beyond the RG motif")
                return dna[:i] + alt + dna[i + len(ref):], i
        # print(iv)
        # print(ref_old)
        # print(ref)
        # print(alt)
        # print(dna, i)
        # print(derived_ref)
        raise ValueError(
            f"REF mismatch at {chrom}:{pos} "
            f"(expected {ref}, found {derived_ref})"
            f"strand: {strand}"
        )

    return dna[:i] + alt + dna[i + len(ref):], i


# --- Load genomic coordinate metadata ----------------------------------------
json_file = "/mnt/d/phd/scripts/14_gnomAD_conservation_proj/data/processed/genomic_coordinates_info_merged_win5_extended.json"
with open(json_file, "r") as f:
    merged_genomic_coordinates_list = json.load(f)

coord_by_region = {
    d["region_id"]: d for d in merged_genomic_coordinates_list
}

# --- Initialize containers ---------------------------------------------------
before_seq = []
after_seq = []
before_dna = []
after_dna = []
mut_pos = []

# --- Apply variants ----------------------------------------------------------
for _, row in df_matched.iterrows():
    region_id = row["region_id"]
    # print(row)
    target = coord_by_region.get(region_id)
    if target is None:
        before_seq.append(None)
        after_seq.append(None)
        before_dna.append(None)
        after_dna.append(None)
        mut_pos.append(None)
        print("no target for region:", region_id)
        continue

    dna = target["dna"]
    intervals = target["intervals"]

    before_dna.append(dna)
    before_seq.append(target["prot_seq"])

    try:
        mutated_dna, idx = apply_variant(
            dna=dna,
            intervals=intervals,
            chrom=row["Chromosome"],
            pos=row["Start"],
            ref=row["REF"],
            alt=row["ALT"],
            strand = row["Strand"],
        )

        after_dna.append(mutated_dna)
        after_seq.append(str(Seq(mutated_dna).translate()))
        mut_pos.append(idx)

    except Exception as e:
        # Hard failure for this variant
        # print("hello:", row["REF"])
        after_dna.append(None)
        after_seq.append(None)
        mut_pos.append(None)
        print(f"[WARN] {region_id}: {e}")

# --- Attach sequence information to DataFrame --------------------------------
df_matched["before_seq"] = before_seq
df_matched["after_seq"] = after_seq
df_matched["before_dna"] = before_dna
df_matched["after_dna"] = after_dna
df_matched["mut_pos"] = mut_pos


# --- Variant classification --------------------------------------------------
df_matched["variant_type"] = [
    varclass.classify_variant(
        row.before_dna,
        row.after_dna,
        row.before_seq,
        row.after_seq
    )
    for row in df_matched.itertuples()
]

# --- RG-specific change metrics ----------------------------------------------
df_rg_changes = df_matched.apply(
    lambda row: varclass.rg_change_from_category(
        category=row["variant_type"],
        before_aa=row["before_seq"],
        after_aa=row["after_seq"],
        mut_pos_dna=row["mut_pos"],
        ref_dna=row["REF"],
        alt_dna=row["ALT"]
    ),
    axis=1
)

df_rg_changes_expanded = pd.json_normalize(df_rg_changes)
df_final = pd.concat([df_matched, df_rg_changes_expanded], axis=1)

# df_final.to_pickle("/mnt/d/phd/scripts/14_gnomAD_conservation_proj/data/output/variant_info_annotated_extended_df.pkl")

# --- Physicochemical metrics -------------------------------------------------
# df_physchem = df_final.apply(
#     lambda row: varclass.get_physchem_metrics_opt(
#         before_aa=row["before_seq"],
#         after_aa=row["after_seq"],
#         category=row["variant_type"]
#     ),
#     axis=1
# )

from multiprocessing import Pool
import os
import pandas as pd

def _worker(args):
    before_aa, after_aa, category = args
    return varclass.get_physchem_metrics_opt(before_aa=before_aa, after_aa=after_aa, category=category)

from tqdm.notebook import tqdm

def run_physchem_parallel(df, n_workers=None):
    n_workers = n_workers or os.cpu_count()
    tuples = list(zip(df["before_seq"], df["after_seq"], df["variant_type"]))
    
    with Pool(n_workers) as pool:
        results = list(tqdm(
            pool.imap(_worker, tuples),
            total=len(tuples),
            desc="Computing physchem metrics"
        ))
    
    return pd.DataFrame(results, index=df.index)

# def run_physchem_parallel(df, n_workers=None):
#     n_workers = n_workers or os.cpu_count()
#     print(n_workers)
#     tuples = list(zip(df["before_seq"], df["after_seq"], df["variant_type"]))
    
#     with Pool(n_workers) as pool:
#         results = pool.map(_worker, tuples)
    
#     return pd.DataFrame(results, index=df.index)

df_physchem = run_physchem_parallel(df_final)

df_physchem_expanded = pd.json_normalize(df_physchem)
df_final = pd.concat([df_final, df_physchem_expanded], axis=1)

# --- Result ------------------------------------------------------------------
print(df_final)

In [ ]:
df_final.to_pickle("/mnt/d/phd/scripts/14_gnomAD_conservation_proj/data/output/variant_info_annotated_extended_df.pkl")

In [ ]:
df_final = pd.read_pickle("/mnt/d/phd/scripts/14_gnomAD_conservation_proj/data/output/variant_info_annotated_extended_df.pkl")
df_final

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as patches

def make_red_white_green():
    reds = plt.cm.Reds_r(np.linspace(0, 1, 128))
    greens = plt.cm.Greens(np.linspace(0, 1, 128))
    center = np.array([[1, 1, 1, 1]])
    colors = np.vstack((reds, center, greens))
    cmap = mcolors.LinearSegmentedColormap.from_list("RedWhiteGreen", colors)
    return cmap

rwgreen = make_red_white_green()

common_cmap_dict = {
    "pos": "Greens",
    "neg": "Reds",
    "mixed": rwgreen
}

aa_groups = {
    "R":"Pos", "K":"Pos", "H":"Pos",
    "D":"Neg", "E":"Neg",
    "S":"Polar", "T":"Polar", "N":"Polar", "Q":"Polar",
    "F":"Aromatic", "W":"Aromatic", "Y":"Aromatic",
    "A":"Hydrophobic", "V":"Hydrophobic", "I":"Hydrophobic",
    "L":"Hydrophobic", "M":"Hydrophobic",
    "G":"C/G/P", "P":"C/G/P", "C":"C/G/P"
}

group_order_cols = ["Aromatic", "C/G/P", "Hydrophobic", "Neg", "Polar", "Pos"]
group_order_rows = group_order_cols[::-1]

ordered_aa = []
for g in group_order_cols:
    ordered_aa.extend(
        sorted([aa for aa, grp in aa_groups.items() if grp == g])
    )
df_missense = (
    df_final[
        (df_final["variant_type"] == "missense") &
        (df_final["before_seq"].notnull()) &
        (df_final["after_seq"].notnull()) &
        (df_final["mut_pos"].notnull()) &
        (df_final["group"].notnull())
    ]
    .copy()
)

def aa_change(row):
    aa_pos = int(row.mut_pos // 3)
    if aa_pos >= len(row.before_seq) or aa_pos >= len(row.after_seq):
        return None
    return f"{row.before_seq[aa_pos]}>{row.after_seq[aa_pos]}"

df_missense["aa_change"] = df_missense.apply(aa_change, axis=1)
df_missense = df_missense[df_missense["aa_change"].notnull()].copy()
df_missense[["AA_from", "AA_to"]] = df_missense["aa_change"].str.split(">", expand=True)

df_pos = df_missense[df_missense["group"] == "positive"]
df_neg = df_missense[df_missense["group"] == "negative"]


group_slices_col = {}

start_idx = 0
for group in group_order_cols:

    aas_in_group = [
        aa for aa in ordered_aa
        if aa_groups[aa] == group
    ]

    if len(aas_in_group) == 0:
        continue

    end_idx = start_idx + len(aas_in_group) - 1

    group_slices_col[group] = (start_idx, end_idx)

    start_idx = end_idx + 1


group_slices_row = {}

start_idx = 0
for group in group_order_rows:

    aas_in_group = [
        aa for aa in ordered_aa
        if aa_groups[aa] == group
    ]

    if len(aas_in_group) == 0:
        continue

    end_idx = start_idx + len(aas_in_group) - 1

    group_slices_row[group] = (start_idx, end_idx)

    start_idx = end_idx + 1


In [ ]:
from scipy.stats import fisher_exact
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as patches
from statsmodels.stats.multitest import multipletests



def make_matrix(df_group, alpha=0.0, return_counts=False):

    counts = (
        df_group
        .groupby(["AA_from", "AA_to"])
        .size()
        .unstack(fill_value=0)
    )

    if return_counts:
        return counts

    # Optional smoothing
    if alpha > 0:
        counts = counts + alpha

    # MLE normalization
    mat = counts.div(counts.sum(axis=1), axis=0)

    # Layout only (anti-diagonal)
    mat = mat.reindex(
        index=ordered_aa[::-1],
        columns=ordered_aa,
        fill_value=0
    )

    return mat

def make_counts(df_group):

    return (
        df_group
        .groupby(["AA_from", "AA_to"])
        .size()
        .unstack(fill_value=0)
    )

def order_mats(mat, order_list, grouped_from = False, grouped_to=False,  fill_value=0):

    if grouped_from:
        return mat.reindex(
            # index=order_list[::-1],
            columns=order_list,
            fill_value=fill_value
        ) 
    if grouped_to:
        return mat.reindex(
            index=order_list[::-1],
            # columns=order_list,
            fill_value=fill_value
        )
    if not grouped_from and not grouped_to:
        return mat.reindex(
            index=order_list[::-1],
            columns=order_list,
            fill_value=fill_value
        )

def normalize_counts(counts, alpha=0.0):

    mat = counts.copy()

    if alpha > 0:
        mat = mat + alpha

    return mat.div(mat.sum(axis=1), axis=0)

def compute_or(mat_pos, mat_neg):

    mat_pos, mat_neg = mat_pos.align(mat_neg, fill_value=0)

    with np.errstate(divide="ignore", invalid="ignore"):

        odds_pos = mat_pos / (1 - mat_pos)
        odds_neg = mat_neg / (1 - mat_neg)

        or_mat = odds_pos / odds_neg
        log2_or = np.log2(or_mat)

    return mat_pos, mat_neg, log2_or




def compute_or_with_fdr(counts_pos, counts_neg, min_total=5, alpha=1):
    """
    Compute log2 odds ratios and FDR-corrected p-values for each substitution.
    Returns normalized matrices, log2 OR, raw p-values, FDR-corrected p-values.
    """
    # Add alpha smoothing
    counts_pos_smooth = counts_pos + alpha
    counts_neg_smooth = counts_neg + alpha

    # Row-normalize
    mat_pos_norm = counts_pos_smooth.div(counts_pos_smooth.sum(axis=1), axis=0)
    mat_neg_norm = counts_neg_smooth.div(counts_neg_smooth.sum(axis=1), axis=0)

    # Align
    mat_pos_norm, mat_neg_norm = mat_pos_norm.align(mat_neg_norm, fill_value=0)

    log2_or = pd.DataFrame(index=mat_pos_norm.index, columns=mat_pos_norm.columns, dtype=float)
    pval_df = pd.DataFrame(index=mat_pos_norm.index, columns=mat_pos_norm.columns, dtype=float)

    # Collect p-values for multiple testing correction
    pvals_list = []
    cells = []

    for aa_from in mat_pos_norm.index:
        row_sum_pos = counts_pos.loc[aa_from].sum()
        row_sum_neg = counts_neg.loc[aa_from].sum()

        for aa_to in mat_pos_norm.columns:
            pos_count = counts_pos.loc[aa_from, aa_to]
            neg_count = counts_neg.loc[aa_from, aa_to]

            # Skip if total counts too low
            if (pos_count + neg_count) < min_total:
                log2_or.loc[aa_from, aa_to] = np.nan
                pval_df.loc[aa_from, aa_to] = np.nan
                continue

            table = np.array([
                [pos_count, row_sum_pos - pos_count],
                [neg_count, row_sum_neg - neg_count]
            ])
            oddsratio, pvalue = fisher_exact(table)
            log2_or.loc[aa_from, aa_to] = np.log2(oddsratio) if oddsratio > 0 else np.nan
            pval_df.loc[aa_from, aa_to] = pvalue

            pvals_list.append(pvalue)
            cells.append((aa_from, aa_to))

    # Multiple testing correction
    # print(pvals_list)
    # print(len(pvals_list))
    reject, pvals_corrected, _, _ = multipletests(pvals_list, method="fdr_bh")
    # pvals_corrected = np.array(pvals_list)
    # print(pvals_corrected)
    fdr_df = pd.DataFrame(index=mat_pos_norm.index, columns=mat_pos_norm.columns, dtype=float)
    for idx, (aa_from, aa_to) in enumerate(cells):
        fdr_df.loc[aa_from, aa_to] = pvals_corrected[idx]

    return mat_pos_norm, mat_neg_norm, log2_or, pval_df, fdr_df

def group_counts(counts, group_from=False, group_to=False):

    df = counts.copy()

    if group_from:
        df["FROM_GROUP"] = df.index.map(aa_groups)
        df = df.groupby("FROM_GROUP").sum()
        df = df.reindex(group_order_rows)

    if group_to:
        df = df.T
        df["TO_GROUP"] = df.index.map(aa_groups)
        df = df.groupby("TO_GROUP").sum()
        df = df.reindex(group_order_cols)
        df = df.T

    return df

def group_matrix(mat, group_from=False, group_to=False):

    df = mat.copy()

    if group_from:
        df["FROM_GROUP"] = df.index.map(aa_groups)
        df = df.groupby("FROM_GROUP").sum()
        df = df.reindex(group_order_rows)

    if group_to:
        df = df.T
        df["TO_GROUP"] = df.index.map(aa_groups)
        df = df.groupby("TO_GROUP").sum()
        df = df.reindex(group_order_cols)
        df = df.T

    return df


def plot_panel(mat_pos, mat_neg, mat_or, title,
                counts_pos=None,
               counts_neg=None,
               group_from=False,
               group_to=False,
               vmax = None,
               diff_abs=None):
    vmax_temp = max(mat_pos.max().max(), mat_neg.max().max())
    if vmax is None:
        vmax = vmax_temp
    else:
        print(vmax_temp)
    diff_abs_temp = np.nanmax(np.abs(mat_or.values[np.isfinite(mat_or.values)]))
    if diff_abs is None:
        diff_abs = diff_abs_temp
    else:
        print(diff_abs_temp)   

    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    # auto annotation
    do_annot = counts_pos is not None
    # --- POSITIVE ---
    sns.heatmap(mat_pos, ax=axes[0],
                cmap=common_cmap_dict["pos"],
                vmin=0, vmax=vmax,
                annot=counts_pos if do_annot else False,
                fmt="d",
                annot_kws={"size":8},
                cbar_kws={"label": "Normalized substitution frequency"})
    axes[0].set_title("Positive")

    # --- NEGATIVE ---
    sns.heatmap(mat_neg, ax=axes[1],
                cmap=common_cmap_dict["neg"],
                vmin=0, vmax=vmax,
                annot=counts_neg if do_annot else False,
                fmt="d",
                annot_kws={"size":8},
                cbar_kws={"label": "Normalized substitution frequency"})
    axes[1].set_title("Negative")

    # --- ENRICHMENT ---
    cmap = common_cmap_dict["mixed"].copy()
    cmap.set_bad(color="gray")

    sns.heatmap(mat_or, ax=axes[2],
                cmap=cmap,
                center=0,
                vmin=-diff_abs,
                vmax=diff_abs,
                cbar_kws={"label": "Log2 Odds Ratio"})
    axes[2].set_title("Enrichment")

    # =====================================================
    # Rectangle Logic
    # =====================================================

    if not (group_from and group_to):

        for ax in axes:

            n_rows, n_cols = mat_pos.shape

            for g, (start, end) in group_slices_col.items():

                # Draw column boxes if TO not grouped
                if not group_to:
                    rect = patches.Rectangle(
                        (start, 0),
                        width=(end - start + 1),
                        height=n_rows,
                        fill=False,
                        edgecolor="black",
                        linewidth=1.2
                    )
                    ax.add_patch(rect)
            for g, (start, end) in group_slices_row.items():   
                # Draw row boxes if FROM not grouped
                if not group_from:
                    rect = patches.Rectangle(
                        (0, start),
                        width=n_cols,
                        height=(end - start + 1),
                        fill=False,
                        edgecolor="black",
                        linewidth=1.2
                    )
                    ax.add_patch(rect)

    fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()



def plot_panel_with_stars(mat_pos, mat_neg, mat_or, fdr_df=None, title="",
                          alpha_star=0.05, counts_pos=None, counts_neg=None,
                          group_from=False, group_to=False, vmax=None, diff_abs=None):
    """
    Plot heatmap panel with Positive, Negative, Enrichment.
    Adds stars on cells passing FDR significance threshold(s).
    """
    vmax_temp = max(mat_pos.max().max(), mat_neg.max().max())
    if vmax is None:
        vmax = vmax_temp
    else:
        print(vmax_temp)
    diff_abs_temp = np.nanmax(np.abs(mat_or.values[np.isfinite(mat_or.values)]))
    if diff_abs is None:
        diff_abs = diff_abs_temp
    else:
        print(diff_abs_temp)   

    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    do_annot = counts_pos is not None

    # --- Positive ---
    sns.heatmap(mat_pos, ax=axes[0],
                cmap=common_cmap_dict["pos"],
                vmin=0, vmax=vmax,
                annot=counts_pos if do_annot else False,
                fmt="d",
                annot_kws={"size":8},
                cbar_kws={"label":"Normalized substitution frequency"})
    axes[0].set_title("Positive")

    # --- Negative ---
    sns.heatmap(mat_neg, ax=axes[1],
                cmap=common_cmap_dict["neg"],
                vmin=0, vmax=vmax,
                annot=counts_neg if do_annot else False,
                fmt="d",
                annot_kws={"size":8},
                cbar_kws={"label":"Normalized substitution frequency"})
    axes[1].set_title("Negative")

    # --- Enrichment (OR) ---
    cmap = common_cmap_dict["mixed"].copy()
    cmap.set_bad(color="gray")
    sns.heatmap(mat_or, ax=axes[2],
                cmap=cmap,
                center=0,
                vmin=-diff_abs,
                vmax=diff_abs,
                cbar_kws={"label": "Log2 Odds Ratio"})
    axes[2].set_title("Enrichment")

    if not (group_from and group_to):

        for ax in axes:

            n_rows, n_cols = mat_pos.shape

            for g, (start, end) in group_slices_col.items():

                # Draw column boxes if TO not grouped
                if not group_to:
                    rect = patches.Rectangle(
                        (start, 0),
                        width=(end - start + 1),
                        height=n_rows,
                        fill=False,
                        edgecolor="black",
                        linewidth=1.2
                    )
                    ax.add_patch(rect)
            for g, (start, end) in group_slices_row.items():   
                # Draw row boxes if FROM not grouped
                if not group_from:
                    rect = patches.Rectangle(
                        (0, start),
                        width=n_cols,
                        height=(end - start + 1),
                        fill=False,
                        edgecolor="black",
                        linewidth=1.2
                    )
                    ax.add_patch(rect)

    # fig.suptitle(title, fontsize=16)
    # plt.tight_layout()
    # plt.show()
    # =====================================================
    # Add stars for significance
    # =====================================================
    if fdr_df is not None:
        n_rows, n_cols = mat_or.shape
        for i, aa_from in enumerate(mat_or.index):
            for j, aa_to in enumerate(mat_or.columns):
                pval = fdr_df.loc[aa_from, aa_to]
                if pd.isna(pval):
                    continue
                # Number of stars: 1 for p<0.05, 2 for p<0.01, 3 for p<0.001
                if pval < 0.001:
                    stars = '***'
                elif pval < 0.01:
                    stars = '**'
                elif pval < alpha_star:
                    stars = '*'
                else:
                    stars = ''
                if stars:
                    axes[2].text(j+0.5, i+0.5, stars,
                                 color='black', ha='center', va='center', fontsize=10)

    fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
# Build matrices (pure MLE)

##### SET SOME SETTINGS

ALPHA = 0
VMAX = None # 0.75
DIFF_ABS = None # 4
MIN_TOTAL = 5
ALPHA_SIG = 0.05  # significance threshold

mat_pos_full = make_matrix(df_pos, alpha=ALPHA)
mat_neg_full = make_matrix(df_neg, alpha=ALPHA)
# print(mat_pos_full)
# OR
mat_pos_full, mat_neg_full, mat_or_full = compute_or(
    mat_pos_full,
    mat_neg_full
)

pos_counts = make_counts(df_pos)
neg_counts = make_counts(df_neg)

pos_counts_ng = order_mats(pos_counts, ordered_aa, grouped_from = False, grouped_to=False, fill_value=0)
neg_counts_ng = order_mats(neg_counts, ordered_aa, grouped_from = False, grouped_to=False, fill_value=0)

# print(pos_counts_ng)
##################################
# Flatten the matrix and get top signals

# signals = []
# for aa_from in mat_or_full.index:
#     for aa_to in mat_or_full.columns:
#         or_value = mat_or_full.loc[aa_from, aa_to]
#         if np.isfinite(or_value):
#             signals.append({
#                 'from': aa_from,
#                 'to': aa_to,
#                 'log2_or': or_value,
#                 'abs_log2_or': abs(or_value)
#             })

# # Convert to dataframe and sort
# signals_df = pd.DataFrame(signals).sort_values('abs_log2_or', ascending=False)

# # Display top 20 strongest signals
# # print("Top 20 strongest signals (positive = enriched in positive group):")
# # print(signals_df.head(20).to_string())
# # Plot top 20 signals as bar graph
# top_20 = signals_df.head(20).sort_values('log2_or')
# fig, ax = plt.subplots(figsize=(10, 6))
# colors = ['red' if x < 0 else 'blue' for x in top_20['log2_or']]
# ax.barh(range(len(top_20)), top_20['log2_or'], color=colors)
# ax.set_yticks(range(len(top_20)))
# ax.set_yticklabels([f"{row['from']} → {row['to']}" for _, row in top_20.iterrows()])
# ax.set_xlabel('Log2 Odds Ratio')
# ax.set_title('Top 20 Strongest Signals')
# ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
# plt.tight_layout()
# plt.show()

####################################
# plot_panel(
#     mat_pos_full,
#     mat_neg_full,
#     mat_or_full,
#     "gnomAD — No Grouping",
#     counts_pos=pos_counts_ng,
#     counts_neg=neg_counts_ng,
#     vmax = VMAX, diff_abs = DIFF_ABS
# )

# Compute OR + FDR
mat_pos_norm_ng, mat_neg_norm_ng, log2_or_ng, pval_df_ng, fdr_df_ng = compute_or_with_fdr(
    pos_counts_ng, neg_counts_ng, min_total=MIN_TOTAL, alpha=ALPHA
)

# Plot with stars
plot_panel_with_stars(
    mat_pos_norm_ng, mat_neg_norm_ng, log2_or_ng, fdr_df=fdr_df_ng,
    title="gnomAD — Enrichment with significance stars",
    alpha_star=ALPHA_SIG,
    counts_pos=pos_counts_ng,
    counts_neg=neg_counts_ng
)




# FROM grouped
# pos_fg = group_matrix(mat_pos_full, group_from=True)
# neg_fg = group_matrix(mat_neg_full, group_from=True)


pos_counts = make_counts(df_pos)
neg_counts = make_counts(df_neg)

pos_counts_fg = group_counts(pos_counts, group_from=True, group_to=False)
neg_counts_fg = group_counts(neg_counts, group_from=True, group_to=False)

pos_mat_fg = normalize_counts(pos_counts_fg, alpha=ALPHA)
neg_mat_fg = normalize_counts(neg_counts_fg, alpha=ALPHA)
# print(pos_mat)
pos_mat_fg = order_mats(pos_mat_fg, ordered_aa, grouped_from = True, grouped_to=False, fill_value=0)
neg_mat_fg = order_mats(neg_mat_fg, ordered_aa, grouped_from = True, grouped_to=False, fill_value=0)

pos_counts_fg = order_mats(pos_counts_fg, ordered_aa, grouped_from = True, grouped_to=False, fill_value=0)
neg_counts_fg = order_mats(neg_counts_fg, ordered_aa, grouped_from = True, grouped_to=False, fill_value=0)

pos_fg, neg_fg, or_fg = compute_or(pos_mat_fg, neg_mat_fg)

# plot_panel(
#     pos_fg, neg_fg, or_fg,
#     "gnomAD — Grouping FROM",
#     counts_pos=pos_counts_fg,
#     counts_neg=neg_counts_fg,
#     group_from=True,
#     vmax = VMAX, diff_abs = DIFF_ABS
# )

# Compute OR + FDR
mat_pos_norm_fg, mat_neg_norm_fg, log2_or_fg, pval_df_fg, fdr_df_fg = compute_or_with_fdr(
    pos_counts_fg, neg_counts_fg, min_total=MIN_TOTAL, alpha=ALPHA
)

# Plot with stars
plot_panel_with_stars(
    mat_pos_norm_fg, mat_neg_norm_fg, log2_or_fg, fdr_df=fdr_df_fg,
    title="gnomAD — Grouping FROM Enrichment with significance stars",
    alpha_star=ALPHA_SIG,
    counts_pos=pos_counts_fg,
    counts_neg=neg_counts_fg,
    group_from=True,
    vmax = VMAX, diff_abs = DIFF_ABS
)

# TO grouped

pos_counts = make_counts(df_pos)
neg_counts = make_counts(df_neg)

pos_counts_tg = group_counts(pos_counts, group_from=False, group_to=True)
neg_counts_tg = group_counts(neg_counts, group_from=False, group_to=True)

pos_mat_tg = normalize_counts(pos_counts_tg, alpha=ALPHA)
neg_mat_tg = normalize_counts(neg_counts_tg, alpha=ALPHA)
# print(pos_mat)
pos_mat_tg = order_mats(pos_mat_tg, ordered_aa, grouped_from = False, grouped_to=True, fill_value=0)
neg_mat_tg = order_mats(neg_mat_tg, ordered_aa, grouped_from = False, grouped_to=True, fill_value=0)

pos_counts_tg = order_mats(pos_counts_tg, ordered_aa, grouped_from = False, grouped_to=True, fill_value=0)
neg_counts_tg = order_mats(neg_counts_tg, ordered_aa, grouped_from = False, grouped_to=True, fill_value=0)
# print(pos_mat)
pos_tg, neg_tg, or_tg = compute_or(pos_mat_tg, neg_mat_tg)

# plot_panel(
#     pos_tg, neg_tg, or_tg,
#     "gnomAD — Grouping TO",
#     group_to=True,
#     vmax = VMAX, diff_abs = DIFF_ABS
# )

# Compute OR + FDR
mat_pos_norm_tg, mat_neg_norm_tg, log2_or_tg, pval_df_tg, fdr_df_tg = compute_or_with_fdr(
    pos_counts_tg, neg_counts_tg, min_total=MIN_TOTAL, alpha=ALPHA
)

# Plot with stars
plot_panel_with_stars(
    mat_pos_norm_tg, mat_neg_norm_tg, log2_or_tg, fdr_df=fdr_df_tg,
    title="gnomAD — Grouping TO Enrichment with significance stars",
    alpha_star=ALPHA_SIG,
    counts_pos=pos_counts_tg,
    counts_neg=neg_counts_tg,
    group_to=True,
    vmax = VMAX, diff_abs = DIFF_ABS
    
)



# BOTH grouped

pos_counts = make_counts(df_pos)
neg_counts = make_counts(df_neg)

pos_counts_bg = group_counts(pos_counts, group_from=True, group_to=True)
neg_counts_bg = group_counts(neg_counts, group_from=True, group_to=True)

pos_mat_bg = normalize_counts(pos_counts_bg, alpha=ALPHA)
neg_mat_bg = normalize_counts(neg_counts_bg, alpha=ALPHA)

pos_bg, neg_bg, or_bg = compute_or(pos_mat_bg, neg_mat_bg)

# plot_panel(
#     pos_bg, neg_bg, or_bg,
#     "gnomAD — Grouping BOTH",
#     group_to=True, group_from=True,
#     vmax = VMAX, diff_abs = DIFF_ABS
# )

# Compute OR + FDR
mat_pos_norm_bg, mat_neg_norm_bg, log2_or_bg, pval_df_bg, fdr_df_bg = compute_or_with_fdr(
    pos_counts_bg, neg_counts_bg, min_total=MIN_TOTAL, alpha=ALPHA
)

# Plot with stars
plot_panel_with_stars(
    mat_pos_norm_bg, mat_neg_norm_bg, log2_or_bg, fdr_df=fdr_df_bg,
    title="gnomAD — Enrichment with significance stars",
    alpha_star=ALPHA_SIG,
    counts_pos=pos_counts_bg,
    counts_neg=neg_counts_bg,
    group_to=True, group_from=True,
    vmax = VMAX, diff_abs = DIFF_ABS
)

# pos_fg = group_matrix(mat_pos_full, group_to=True, group_from=True)
# neg_fg = group_matrix(mat_neg_full, group_to=True, group_from=True)

# pos_fg, neg_fg, or_fg = compute_or(pos_fg, neg_fg)

# plot_panel(
#     pos_fg, neg_fg, or_fg,
#     "gnomAD — Grouping BOTH",
#     group_to=True, group_from=True
# )

In [ ]:
import pandas as pd
# Flatten the log2_or_ng matrix to find top signals, excluding infs and filtering by p-value < 0.05
signals = []
for aa_from in log2_or_ng.index:
    for aa_to in log2_or_ng.columns:
        or_value = log2_or_ng.loc[aa_from, aa_to]
        pval = fdr_df_ng.loc[aa_from, aa_to]
        if not pd.isna(or_value) and not np.isinf(or_value) and not pd.isna(pval) and pval < 0.001:
            signals.append({
                'from': aa_from,
                'to': aa_to,
                'log2_or': or_value,
                'abs_log2_or': abs(or_value),
                'p_value': pval
            })

# Convert to DataFrame and sort by absolute log2 OR descending
signals_df = pd.DataFrame(signals).sort_values('abs_log2_or', ascending=False)

# Get top 10
top_10 = signals_df.head(10)
print(top_10)

In [ ]:
###### BACKUP

In [ ]:
# --- Imports and path setup ---------------------------------------------------
import os
import sys
import json
import pandas as pd
from importlib import reload
from Bio.Seq import Seq

# Add project root (one level above notebooks/) to sys.path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import variant_classification as varclass
reload(varclass)

# --- Load genomic coordinate metadata ----------------------------------------
json_file = "/mnt/d/phd/scripts/14_gnomAD_conservation_proj/data/processed/genomic_coordinates_info_merged.json"
with open(json_file, "r") as f:
    merged_genomic_coordinates_list = json.load(f)

# --- Initialize containers ---------------------------------------------------
before_seq = []
after_seq = []
before_dna = []
after_dna = []
mut_pos = []

# --- Construct before/after DNA and protein sequences ------------------------
for _, row in df_matched[df_matched["region_id"] == "Q9UPT8_235_271"].iterrows():
    region_id = row["region_id"]

    target = next(
        (d for d in merged_genomic_coordinates_list if d["region_id"] == region_id),
        None
    )
    # print(target)
    # Store original sequences
    before_seq.append(target["prot_seq"])
    curr_before_dna = target["dna"]
    before_dna.append(curr_before_dna)

    curr_result = None
    curr_transl_change = None
    curr_start_of_orig = None

    if len(target["intervals"]) == 1:
        interval = target["intervals"][0]

        curr_start_of_orig = row["Start"] - interval["start"]
        curr_end_of_change = curr_start_of_orig + len(row["REF"])

        curr_result = (
            curr_before_dna[:curr_start_of_orig]
            + row["ALT"]
            + curr_before_dna[curr_end_of_change:]
        )
        curr_transl_change = str(Seq(curr_result).translate())

    elif len(target["intervals"]) == 0:
        raise ValueError(f"Region {region_id} has zero intervals (unexpected).")
    elif len(target["intervals"]) > 1:
        print("MULTI-INTERVAL CASE")
        flag_interval_found = False
        curr_seq_position_in_dna = 0
        curr_result = ""
        ### check in which interval the change happens
        if flag_interval_found:
            curr_result += curr_before_dna[:len_of_interval]
        for inval in target["intervals"]:
            ## calc posistions of the current interval in the dna seq
            curr_start_of_orig = row["Start"] - inval["start"]
            len_of_interval = inval["end"] - inval["start"]
            if curr_start_of_orig > len_of_interval:
                #### the muation is not in this interval
                curr_result += curr_before_dna[:len_of_interval]
            else:
                ##### the mutation is in this interval
                curr_end_of_change = curr_start_of_orig + len(row["REF"])
                curr_result = (
                    curr_before_dna[:curr_start_of_orig]
                    + row["ALT"]
                    + curr_before_dna[curr_end_of_change:]
                    )
                flag_interval_found = True
            curr_seq_position_in_dna += len_of_interval
        curr_transl_change = str(Seq(curr_result).translate())
    print(target["dna"])
    print(target["prot_seq"])
    print(row["REF"])
    print(row["ALT"])
    print(curr_result)
    print(curr_transl_change)
    print(curr_start_of_orig)
    print("________________________________")
        
        #### this is for the cases with multiple intervals
    # For now, multi-interval cases are left as None
    after_dna.append(curr_result)
    after_seq.append(curr_transl_change)
    mut_pos.append(curr_start_of_orig)

# --- Attach sequence information to DataFrame --------------------------------
df_matched["before_seq"] = before_seq
df_matched["after_seq"] = after_seq
df_matched["before_dna"] = before_dna
df_matched["after_dna"] = after_dna
df_matched["mut_pos"] = mut_pos


In [ ]:
df_final.head()

In [ ]:
import pandas as pd
import sys
import os
import json
from Bio.Seq import Seq
# Add the project root (one level above notebooks/) to sys.path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)
# print(project_root)

import variant_classification as varclass
from importlib import reload

# Reload the module after making changes
reload(varclass)

before_seq, after_seq = [], []
before_dna, after_dna = [], []
mut_pos = []
json_file = "/mnt/d/phd/scripts/14_gnomAD_conservation_proj/data/processed/genomic_coordinates_info_merged.json"
with open(json_file, "r") as f:
    merged_genomic_coordinates_list = json.load(f)

for i, el in df_matched.iterrows():
    curr_region_id = el["region_id"]

    target = next((d for d in merged_genomic_coordinates_list if d["region_id"] == curr_region_id), None)
    # print(target)
    before_seq.append(target['prot_seq'])
    curr_before_dna = target['dna']
    before_dna.append(curr_before_dna)
    if len(target['intervals']) >1:
        # print("this one has mutliple cuts, lets work on this later")
        curr_transl_change = None
        curr_result = None
        curr_start_of_orig = None
    elif len(target['intervals']) == 1:
        ### best use case to work on
        curr_start_of_orig = el["Start"] - target['intervals'][0]["start"]
        curr_end_of_change = curr_start_of_orig + len(el["REF"])
        curr_dna_change = el["ALT"]
        curr_result = curr_before_dna[:curr_start_of_orig] + curr_dna_change + curr_before_dna[curr_end_of_change:]
        curr_transl_change = str(Seq(curr_result).translate())
        # print(curr_transl_change)
    else:
        print("This entry has 0 intervals!!!! How?? This should be impossible.")
    after_dna.append(curr_result)
    mut_pos.append(curr_start_of_orig)
    #### find position of the change
    # find the right interval 
    # right now skip ones that are overlapping in multiple
    # this minus the start of the interval is the start value, then exchange the old one to the new one and translate

    after_seq.append(curr_transl_change)
df_matched["before_seq"] = before_seq
df_matched["after_seq"] = after_seq

df_matched["before_dna"] = before_dna
df_matched["after_dna"] = after_dna


df_matched["mut_pos"] = mut_pos


# out = df_matched.apply(lambda row: get_RG_metrics(row.before_seq, row.after_seq), axis=1)
# print(out)
# Step 2 — convert each dict into columns
# out_df = pd.DataFrame(list(out))

# df_matched = df_matched.join(out)
# Step 1 — apply, returns a Series where each cell is a dict
# out = df_matched.apply(
#     lambda row: get_RG_metrics(row.before_seq, row.after_seq),
#     axis=1
# )

df_matched["variant_type"] = [
    varclass.classify_variant(row.before_dna, row.after_dna, row.before_seq, row.after_seq)
    for row in df_matched.itertuples()
]


df_results = df_matched.apply(
    lambda row: varclass.rg_change_from_category(
        category=row['variant_type'],
        before_aa=row['before_seq'],
        after_aa=row['after_seq'],
        mut_pos_dna=row['mut_pos'],
        ref_dna=row['REF'],
        alt_dna=row['ALT']
    ), axis=1
)

# df_results is a Series of dicts. Expand into separate columns
df_expanded = pd.json_normalize(df_results)

# Combine with original DataFrame
df_final = pd.concat([df_matched, df_expanded], axis=1)

# print(df_final)


df_metrics = df_final.apply(
    lambda row: varclass.get_physchem_metrics(
        before_aa=row['before_seq'],
        after_aa=row['after_seq'],
        category = row['variant_type']
    ),
    axis=1
)

# Expand the resulting dictionaries into columns
df_metrics_expanded = pd.json_normalize(df_metrics)

# Combine with the original DataFrame
df_final2 = pd.concat([df_final, df_metrics_expanded], axis=1)

print(df_final2)
# out = df_matched.apply(lambda r: rg_change_from_category(r.before_seq, r.after_seq), axis=1)
# empty = {k: None for k in out.dropna().iloc[0].keys()}
# out_df = out.apply(lambda x: x if isinstance(x, dict) else empty).apply(pd.Series)
# df_matched = df_matched.join(out_df)

# # Step 3 — join safely
# df_matched = df_matched.join(out_df)




#### test

# print(varclass.classify_variant("CCCCGCGGGGAAGCTCCTGGCCCCGGGAGACGGGGG", "CCTCGCGGGGAAGCTCCTGGCCCCGGGAGACGGGGG", "PRGEAPGPGRRG", "PRGEAPGPGRRG"))
# print(varclass.classify_variant("CCCCGCGGGGAAGCTCCTGGCCCCGGGAGACGGGGG", "CCCAGCGGGGAAGCTCCTGGCCCCGGGAGACGGGGG", "PRGEAPGPGRRG", "PSGEAPGPGRRG"))
# print(varclass.classify_variant("AAGCGAGGGAGTGTGAGCAGGGGC", "AAGTGAGGGAGTGTGAGCAGGGGC", "KRGSVSRG", "K*GSVSRG"))
# print(varclass.classify_variant("TCCCGGGGAGGCGGAGGGGGATCCCGCGGGGGC", "TCCCGGGAGGCGGAGGGGGATCCCGCGGGGGC", "SRGGGGGSRGG", "SREAEGDPAG"))
# print(varclass.classify_variant("CATCGAGGCGGCGGGGAGCCCCGCGGGGGC", "CATCGAGGCGGCGGGGAGCCCCGCGGG", "HRGGGEPRGG", "HRGGGEPRG"))



# print(varclass.rg_change_from_category("missense", "PRGEAPGPGRRG", "LRGEAPGPGRRG", 1, "C", "T"))
# print(varclass.rg_change_from_category("missense", "PRGEAPGPGRRG", "PSGEAPGPGRRG", 3, "C", "A"))
# print(varclass.rg_change_from_category("frameshift", "PRGEAPGPGRRG", "PRGKLLAPGDG", 4, "GC", "G"))
# print(varclass.rg_change_from_category("inframe_insertion", "SRGGGGGSRGG", "SRGGGGGSRGGGSRGG",9, "G", "GGCGGAGGGGGATCCC"))

In [ ]:
from typing import Literal

VariantType = Literal[
    "silent",
    "missense",
    "nonsense",
    "inframe_insertion",
    "inframe_deletion",
    "complex",
    "frameshift"
]


def classify_variant(before_dna: str, after_dna: str,
                     before_aa: str, after_aa: str) -> VariantType:
    """
    Classify a protein/DNA change into mutation types:
    silent, missense, nonsense, inframe_insertion, 
    inframe_deletion, complex, frameshift.

    Assumes coding DNA and correct translation.
    """
    if after_dna is None:
        return None
    if after_aa is None:
        return None
    if before_aa == after_aa:
        return "silent"

    # 3. Same AA length → could be silent, missense, nonsense, complex
    # Compare residue by residue
    diffs = [(i, a, b) for i, (a, b) in enumerate(zip(before_aa, after_aa), 1) if a != b]

    # 4. If any change introduces a stop codon
    if any(new == "*" for _, _, new in diffs):
        return "nonsense"

    # 5. Single AA change → missense
    if len(diffs) == 1 and len(before_aa)==len(after_aa):
        return "missense"
    # 1. Frameshift check (DNA length change not multiple of 3)
    dna_len_change = len(after_dna) - len(before_dna)
    if dna_len_change % 3 != 0:
        return "frameshift"

    # 2. In-frame insertion or deletion (no frameshift)
    aa_len_change = len(after_aa) - len(before_aa)

    if aa_len_change > 0:
        return "inframe_insertion"
    elif aa_len_change < 0:
        return "inframe_deletion"
    # 6. Multiple AA changes but no frameshift → complex substitution
    return "complex"


print(classify_variant("CCCCGCGGGGAAGCTCCTGGCCCCGGGAGACGGGGG", "CCTCGCGGGGAAGCTCCTGGCCCCGGGAGACGGGGG", "PRGEAPGPGRRG", "PRGEAPGPGRRG"))
print(classify_variant("CCCCGCGGGGAAGCTCCTGGCCCCGGGAGACGGGGG", "CCCAGCGGGGAAGCTCCTGGCCCCGGGAGACGGGGG", "PRGEAPGPGRRG", "PSGEAPGPGRRG"))
print(classify_variant("AAGCGAGGGAGTGTGAGCAGGGGC", "AAGTGAGGGAGTGTGAGCAGGGGC", "KRGSVSRG", "K*GSVSRG"))
print(classify_variant("TCCCGGGGAGGCGGAGGGGGATCCCGCGGGGGC", "TCCCGGGAGGCGGAGGGGGATCCCGCGGGGGC", "SRGGGGGSRGG", "SREAEGDPAG"))
print(classify_variant("CATCGAGGCGGCGGGGAGCCCCGCGGGGGC", "CATCGAGGCGGCGGGGAGCCCCGCGGG", "HRGGGEPRGG", "HRGGGEPRG"))



# for el in ["CATCGAGGCGGCGGGGAGCCCCGCGGGGGC", "CATCGAGGCGGCGGGGAGCCCCGCGGG"]:
#     print(str(Seq(el).translate()))
#     print(len(el))

In [ ]:
import re
from localcider.sequenceParameters import SequenceParameters as SeqParams

def get_physchem_metrics(before_aa: str, after_aa: str, category: str):
    

    # ---------- per-residue property changes ----------
    # charge_change = 0
    # polarity_change = 0
    # aromatic_change = 0
    # print(before_aa)
    # print(after_aa)
    # print(category)
    if category == "nonsense" or before_aa is None or after_aa is None or '*' in after_aa or after_aa is "":
        return {
        "NCPR_change": None,
        "FCR_change": None,
        "hydropathy_change": None,
        "kappa_change": None,
        "pos_count_change": None,
        "neg_count_change": None,
        "aromaticity_change": None
        }
    # align by min length (simple assumption; this is intentional)
    # L = min(len(before_aa), len(after_aa))

    NCPR_change = SeqParams(after_aa).get_NCPR() - SeqParams(before_aa).get_NCPR()
    FCR_change = SeqParams(after_aa).get_FCR() - SeqParams(before_aa).get_FCR()
    hydropathy_change = SeqParams(after_aa).get_mean_hydropathy() - SeqParams(before_aa).get_mean_hydropathy()
    kappa_change = SeqParams(after_aa).get_kappa() - SeqParams(before_aa).get_kappa()
    pos_count_change = SeqParams(after_aa).get_countPos() - SeqParams(before_aa).get_countPos()
    neg_count_change = SeqParams(after_aa).get_countNeg() - SeqParams(before_aa).get_countNeg()
    aromaticity_change =    ((SeqParams(after_aa).get_amino_acid_fractions()["Y"] + 
                            SeqParams(after_aa).get_amino_acid_fractions()["F"] + 
                            SeqParams(after_aa).get_amino_acid_fractions()["W"]) - 
                            (SeqParams(before_aa).get_amino_acid_fractions()["Y"] + 
                            SeqParams(before_aa).get_amino_acid_fractions()["F"] + 
                            SeqParams(before_aa).get_amino_acid_fractions()["W"] ))


    return {

        "NCPR_change": NCPR_change,
        "FCR_change": FCR_change,
        "hydropathy_change": hydropathy_change,
        "kappa_change": kappa_change,
        "pos_count_change": pos_count_change,
        "neg_count_change": neg_count_change,
        "aromaticity_change": aromaticity_change
    }


def count_RG_positions(seq):
    """Return all start positions of 'RG' motifs (0-based)."""
    return [m.start() for m in re.finditer("RG", seq)]


def rg_change_from_category(category, before_aa, after_aa,
                            mut_pos_dna, ref_dna, alt_dna):
    """
    category: one of
        'silent', 'missense', 'inframe_indel', 'frameshift',
        'nonsense', 'noncoding'
    before_aa: AA sequence before mutation
    after_aa: AA sequence after mutation (None if not applicable)
    mut_pos_dna: 1-based genomic DNA start position of the REF allele
    ref_dna, alt_dna: provided but only needed for frameshift logic

    Returns:
        {
            'rg_before': [...],
            'rg_after': [... or None],
            'gained': int,
            'lost': int,
            'unchanged': int
        }
    """
    # if category == None:
    #     return {
    #         'category': category,
    #         'rg_before': rg_before,
    #         'rg_after': rg_after,
    #         'gained': len(new),
    #         'lost': len(lost),
    #         'unchanged': len(unchanged)
    #     }


    # Count RG before
    rg_before = count_RG_positions(before_aa)

    # # ==============================================================
    # # CASE 1 — Noncoding variants
    # # ==============================================================
    # if category == "noncoding":
    #     return {
    #         'category': category,
    #         'rg_before': rg_before,
    #         'rg_after': rg_before,
    #         'gained': 0,
    #         'lost': 0,
    #         'unchanged': len(rg_before)
    #     }

    # ==============================================================
    # CASE 2 — Silent variants
    # (AA sequences identical)
    # ==============================================================
    if category in ("silent", None):
        return {
            # 'category': category,
            'rg_before': rg_before,
            'rg_after': rg_before,
            'gained': 0,
            'lost': 0,
            'unchanged': len(rg_before)
        }

    # ==============================================================
    # CASE 3 — Missense / In-frame Indels / Nonsense
    # (Direct AA comparison)
    # ==============================================================
    if category in ("missense", "nonsense",  "inframe_insertion", "inframe_deletion"):
        rg_after = count_RG_positions(after_aa)

        lost = len([pos for pos in rg_before if pos not in rg_after])
        gained = len([pos for pos in rg_after if pos not in rg_before])
        unchanged = len(rg_before) - lost

        return {
            # 'category': category,
            'rg_before': rg_before,
            'rg_after': rg_after,
            'gained': gained,
            'lost': lost,
            'unchanged': unchanged
        }

    # ==============================================================
    # CASE 4 — Frameshift
    # Only RG motifs upstream of the mutation position remain valid.
    # ==============================================================
    if category == "frameshift":
        aa_mut_pos = mut_pos_dna // 3
        # print(mut_pos_dna)
        # print(aa_mut_pos)
        rg_after = count_RG_positions(after_aa)

        # RGs before the mutation are unchanged
        unchanged = [pos for pos in rg_before if pos < aa_mut_pos]

        # RGs in the original sequence that overlap or are after the mutation are lost
        lost = [pos for pos in rg_before if pos >= aa_mut_pos]

        # RGs in the mutated sequence that are at or after the mutation are new
        new = [pos for pos in rg_after if pos >= aa_mut_pos]

        return {
            'rg_before': rg_before,
            'rg_after': rg_after,
            'gained': len(new),
            'lost': len(lost),
            'unchanged': len(unchanged)
        }

    # ==============================================================
    # Unknown category
    # ==============================================================
    # print(category)
    raise ValueError(f"Unknown category: {category}")

print(rg_change_from_category("missense", "PRGEAPGPGRRG", "LRGEAPGPGRRG", 1, "C", "T"))
print(rg_change_from_category("missense", "PRGEAPGPGRRG", "PSGEAPGPGRRG", 3, "C", "A"))
print(rg_change_from_category("frameshift", "PRGEAPGPGRRG", "PRGKLLAPGDG", 4, "GC", "G"))
print(rg_change_from_category("inframe_insertion", "SRGGGGGSRGG", "SRGGGGGSRGGGSRGG",9, "G", "GGCGGAGGGGGATCCC"))
# print(rg_change_from_category("missense", "PRGEAPGPGRRG", "PSGEAPGPGRRG", 3, "C", "A"))


In [ ]:
### for each entry in df-overlap, get a before and after sequence
### then do things with it
### count RGs lost (remember position of RGs before and compare with the after sequence) 

### make a column that is len(alt)/len(REF) and direction (F or R)
### check how many variants i have that are divisble by 3, where i can apply a 1to1 variant search


# from typing import Literal

# VariantType = Literal[
#     "silent",
#     "missense",
#     "nonsense",
#     "inframe_insertion",
#     "inframe_deletion",
#     "complex",
#     "frameshift"
# ]


# def classify_variant(before_dna: str, after_dna: str,
#                      before_aa: str, after_aa: str) -> VariantType:
#     """
#     Classify a protein/DNA change into mutation types:
#     silent, missense, nonsense, inframe_insertion, 
#     inframe_deletion, complex, frameshift.

#     Assumes coding DNA and correct translation.
#     """
#     if after_dna is None:
#         return None
#     if after_aa is None:
#         return None
#     if before_aa == after_aa:
#         return "silent"

#     # 3. Same AA length → could be silent, missense, nonsense, complex
#     # Compare residue by residue
#     diffs = [(i, a, b) for i, (a, b) in enumerate(zip(before_aa, after_aa), 1) if a != b]

#     # 4. If any change introduces a stop codon
#     if any(new == "*" for _, _, new in diffs):
#         return "nonsense"

#     # 5. Single AA change → missense
#     if len(diffs) == 1 and len(before_aa)==len(after_aa):
#         return "missense"
#     # 1. Frameshift check (DNA length change not multiple of 3)
#     dna_len_change = len(after_dna) - len(before_dna)
#     if dna_len_change % 3 != 0:
#         return "frameshift"

#     # 2. In-frame insertion or deletion (no frameshift)
#     aa_len_change = len(after_aa) - len(before_aa)

#     if aa_len_change > 0:
#         return "inframe_insertion"
#     elif aa_len_change < 0:
#         return "inframe_deletion"
#     # 6. Multiple AA changes but no frameshift → complex substitution
#     return "complex"

# import re
# from localcider.sequenceParameters import SequenceParameters as SeqParams

# def get_physchem_metrics(before_aa: str, after_aa: str, category: str):
    

#     # ---------- per-residue property changes ----------
#     # charge_change = 0
#     # polarity_change = 0
#     # aromatic_change = 0
#     # print(before_aa)
#     # print(after_aa)
#     # print(category)
#     if category == "nonsense" or before_aa is None or after_aa is None or '*' in after_aa or after_aa is "":
#         return {
#         "NCPR_change": None,
#         "FCR_change": None,
#         "hydropathy_change": None,
#         "kappa_change": None,
#         "pos_count_change": None,
#         "neg_count_change": None,
#         "aromaticity_change": None
#         }
#     # align by min length (simple assumption; this is intentional)
#     # L = min(len(before_aa), len(after_aa))

#     NCPR_change = SeqParams(after_aa).get_NCPR() - SeqParams(before_aa).get_NCPR()
#     FCR_change = SeqParams(after_aa).get_FCR() - SeqParams(before_aa).get_FCR()
#     hydropathy_change = SeqParams(after_aa).get_mean_hydropathy() - SeqParams(before_aa).get_mean_hydropathy()
#     kappa_change = SeqParams(after_aa).get_kappa() - SeqParams(before_aa).get_kappa()
#     pos_count_change = SeqParams(after_aa).get_countPos() - SeqParams(before_aa).get_countPos()
#     neg_count_change = SeqParams(after_aa).get_countNeg() - SeqParams(before_aa).get_countNeg()
#     aromaticity_change =    ((SeqParams(after_aa).get_amino_acid_fractions()["Y"] + 
#                             SeqParams(after_aa).get_amino_acid_fractions()["F"] + 
#                             SeqParams(after_aa).get_amino_acid_fractions()["W"]) - 
#                             (SeqParams(before_aa).get_amino_acid_fractions()["Y"] + 
#                             SeqParams(before_aa).get_amino_acid_fractions()["F"] + 
#                             SeqParams(before_aa).get_amino_acid_fractions()["W"] ))


#     return {

#         "NCPR_change": NCPR_change,
#         "FCR_change": FCR_change,
#         "hydropathy_change": hydropathy_change,
#         "kappa_change": kappa_change,
#         "pos_count_change": pos_count_change,
#         "neg_count_change": neg_count_change,
#         "aromaticity_change": aromaticity_change
#     }


# def count_RG_positions(seq):
#     """Return all start positions of 'RG' motifs (0-based)."""
#     return [m.start() for m in re.finditer("RG", seq)]


# def rg_change_from_category(category, before_aa, after_aa,
#                             mut_pos_dna, ref_dna, alt_dna):
#     """
#     category: one of
#         'silent', 'missense', 'inframe_indel', 'frameshift',
#         'nonsense', 'noncoding'
#     before_aa: AA sequence before mutation
#     after_aa: AA sequence after mutation (None if not applicable)
#     mut_pos_dna: 1-based genomic DNA start position of the REF allele
#     ref_dna, alt_dna: provided but only needed for frameshift logic

#     Returns:
#         {
#             'rg_before': [...],
#             'rg_after': [... or None],
#             'gained': int,
#             'lost': int,
#             'unchanged': int
#         }
#     """
#     # if category == None:
#     #     return {
#     #         'category': category,
#     #         'rg_before': rg_before,
#     #         'rg_after': rg_after,
#     #         'gained': len(new),
#     #         'lost': len(lost),
#     #         'unchanged': len(unchanged)
#     #     }


#     # Count RG before
#     rg_before = count_RG_positions(before_aa)

#     # # ==============================================================
#     # # CASE 1 — Noncoding variants
#     # # ==============================================================
#     # if category == "noncoding":
#     #     return {
#     #         'category': category,
#     #         'rg_before': rg_before,
#     #         'rg_after': rg_before,
#     #         'gained': 0,
#     #         'lost': 0,
#     #         'unchanged': len(rg_before)
#     #     }

#     # ==============================================================
#     # CASE 2 — Silent variants
#     # (AA sequences identical)
#     # ==============================================================
#     if category in ("silent", None):
#         return {
#             # 'category': category,
#             'rg_before': rg_before,
#             'rg_after': rg_before,
#             'gained': 0,
#             'lost': 0,
#             'unchanged': len(rg_before)
#         }

#     # ==============================================================
#     # CASE 3 — Missense / In-frame Indels / Nonsense
#     # (Direct AA comparison)
#     # ==============================================================
#     if category in ("missense", "nonsense",  "inframe_insertion", "inframe_deletion"):
#         rg_after = count_RG_positions(after_aa)

#         lost = len([pos for pos in rg_before if pos not in rg_after])
#         gained = len([pos for pos in rg_after if pos not in rg_before])
#         unchanged = len(rg_before) - lost

#         return {
#             # 'category': category,
#             'rg_before': rg_before,
#             'rg_after': rg_after,
#             'gained': gained,
#             'lost': lost,
#             'unchanged': unchanged
#         }

#     # ==============================================================
#     # CASE 4 — Frameshift
#     # Only RG motifs upstream of the mutation position remain valid.
#     # ==============================================================
#     if category == "frameshift":
#         aa_mut_pos = mut_pos_dna // 3
#         # print(mut_pos_dna)
#         # print(aa_mut_pos)
#         rg_after = count_RG_positions(after_aa)

#         # RGs before the mutation are unchanged
#         unchanged = [pos for pos in rg_before if pos < aa_mut_pos]

#         # RGs in the original sequence that overlap or are after the mutation are lost
#         lost = [pos for pos in rg_before if pos >= aa_mut_pos]

#         # RGs in the mutated sequence that are at or after the mutation are new
#         new = [pos for pos in rg_after if pos >= aa_mut_pos]

#         return {
#             'rg_before': rg_before,
#             'rg_after': rg_after,
#             'gained': len(new),
#             'lost': len(lost),
#             'unchanged': len(unchanged)
#         }

#     # ==============================================================
#     # Unknown category
#     # ==============================================================
#     # print(category)
#     raise ValueError(f"Unknown category: {category}")





before_seq, after_seq = [], []
before_dna, after_dna = [], []
mut_pos = []

for i, el in df_overlap.iterrows():
    curr_region_id = el["region_id"]

    target = next((d for d in merged_genomic_coordinates_list if d["region_id"] == curr_region_id), None)
    # print(target)
    before_seq.append(target['prot_seq'])
    curr_before_dna = target['dna']
    before_dna.append(curr_before_dna)
    if len(target['intervals']) >1:
        # print("this one has mutliple cuts, lets work on this later")
        curr_transl_change = None
        curr_result = None
        curr_start_of_orig = None
    elif len(target['intervals']) == 1:
        ### best use case to work on
        curr_start_of_orig = el["Start"] - target['intervals'][0]["start"]
        curr_end_of_change = curr_start_of_orig + len(el["REF"])
        curr_dna_change = el["ALT"]
        curr_result = curr_before_dna[:curr_start_of_orig] + curr_dna_change + curr_before_dna[curr_end_of_change:]
        curr_transl_change = str(Seq(curr_result).translate())
        # print(curr_transl_change)
    else:
        print("This entry has 0 intervals!!!! How?? This should be impossible.")
    after_dna.append(curr_result)
    mut_pos.append(curr_start_of_orig)
    #### find position of the change
    # find the right interval 
    # right now skip ones that are overlapping in multiple
    # this minus the start of the interval is the start value, then exchange the old one to the new one and translate

    after_seq.append(curr_transl_change)
df_overlap["before_seq"] = before_seq
df_overlap["after_seq"] = after_seq

df_overlap["before_dna"] = before_dna
df_overlap["after_dna"] = after_dna


df_overlap["mut_pos"] = mut_pos


# out = df_overlap.apply(lambda row: get_RG_metrics(row.before_seq, row.after_seq), axis=1)
# print(out)
# Step 2 — convert each dict into columns
# out_df = pd.DataFrame(list(out))

# df_overlap = df_overlap.join(out)
# Step 1 — apply, returns a Series where each cell is a dict
# out = df_overlap.apply(
#     lambda row: get_RG_metrics(row.before_seq, row.after_seq),
#     axis=1
# )

df_overlap["variant_type"] = [
    classify_variant(row.before_dna, row.after_dna, row.before_seq, row.after_seq)
    for row in df_overlap.itertuples()
]


df_results = df_overlap.apply(
    lambda row: rg_change_from_category(
        category=row['variant_type'],
        before_aa=row['before_seq'],
        after_aa=row['after_seq'],
        mut_pos_dna=row['mut_pos'],
        ref_dna=row['REF'],
        alt_dna=row['ALT']
    ), axis=1
)

# df_results is a Series of dicts. Expand into separate columns
df_expanded = pd.json_normalize(df_results)

# Combine with original DataFrame
df_final = pd.concat([df_overlap, df_expanded], axis=1)

# print(df_final)


df_metrics = df_final.apply(
    lambda row: get_physchem_metrics(
        before_aa=row['before_seq'],
        after_aa=row['after_seq'],
        category = row['variant_type']
    ),
    axis=1
)

# Expand the resulting dictionaries into columns
df_metrics_expanded = pd.json_normalize(df_metrics)

# Combine with the original DataFrame
df_final2 = pd.concat([df_final, df_metrics_expanded], axis=1)

print(df_final2)
# out = df_overlap.apply(lambda r: rg_change_from_category(r.before_seq, r.after_seq), axis=1)
# empty = {k: None for k in out.dropna().iloc[0].keys()}
# out_df = out.apply(lambda x: x if isinstance(x, dict) else empty).apply(pd.Series)
# df_overlap = df_overlap.join(out_df)

# # Step 3 — join safely
# df_overlap = df_overlap.join(out_df)



In [ ]:
##### BACKUP 


import re
# as before we import the SequenceParameters class directly



# print(SeqParams("RRDDAARR").get_NCPR())
# print(SeqParams("RRDAA").get_FCR())
# print(SeqParams("RRDDAARR").get_mean_net_charge())
# print(SeqParams("RRDDYFWAARR").get_amino_acid_fractions()["Y"])

# def find_RG(seq: str):
#     return [m.start() for m in re.finditer(r"RG", seq)]

# def find_RGG(seq: str):
#     return [m.start() for m in re.finditer(r"RG", seq)]

# def charge_of(aa: str):
#     """Return numeric charge: +1, -1, 0. '*' and '-' return None."""
#     if aa in "KRH": return 1
#     if aa in "DE":  return -1
#     if aa in "-*":  return None
#     return 0

# def is_polar(aa: str):
#     # rough polar set: N, Q, S, T, C, Y
#     return aa in "NQSTCY"

# def is_aromatic(aa: str):
#     return aa in "FWYH"   # H included: aromatic ring

# def classify_RG_changes(before: str, after: str, ref: str, alt: str, mut_pos: int):
#     """
#     before  : original AA sequence
#     after   : mutated AA sequence
#     ref     : reference AAs replaced (e.g. 'QK')
#     alt     : replacement AAs (e.g. 'RG')
#     mut_pos : 0-based index of where REF begins in 'before'
#     """

#     # RG positions in both sequences
#     RG_before = find_RG(before)
#     RG_after = find_RG(after)

#     # Net length difference
#     Δ = len(alt) - len(ref)

#     # Convenient ranges
#     ref_start = mut_pos
#     ref_end = mut_pos + len(ref)     # exclusive
#     alt_end = mut_pos + len(alt)

#     untouched_RG = []
#     lost_RG = []
#     new_RG = []

#     # 1. RGs *before* mutation region → direct positional comparison
#     for pos in RG_before:
#         if pos + 1 < ref_start:  # RG spans positions pos,pos+1
#             if pos in RG_after:
#                 untouched_RG.append(pos)
#             else:
#                 lost_RG.append(pos)

#     # 2. RGs fully *inside* the mutation region → compare via coordinate mapping
#     for pos in RG_before:
#         if ref_start <= pos < ref_end:  
#             # Map position in 'before' to corresponding relative position in ALT
#             rel_pos = pos - ref_start
#             # Only count as untouched if ALT also has RG at this relative match
#             if 0 <= rel_pos < len(alt) - 1 and alt[rel_pos:rel_pos+2] == "RG":
#                 # The mapped absolute position in 'after'
#                 mapped_after_pos = ref_start + rel_pos
#                 if mapped_after_pos in RG_after:
#                     untouched_RG.append(pos)
#                 else:
#                     lost_RG.append(pos)
#             else:
#                 lost_RG.append(pos)

#     # 3. RGs *after* the mutation region → shifted positions
#     for pos in RG_before:
#         if pos >= ref_end:
#             mapped = pos + Δ
#             if mapped in RG_after:
#                 untouched_RG.append(pos)
#             else:
#                 lost_RG.append(pos)

#     # Now detect NEW RGs that didn't exist before
#     # Map after→before and see which have no corresponding origin
#     before_positions_possible = set()

#     # Reverse mapping for after positions
#     for pos in RG_after:
#         # Case: before mutation region
#         if pos + 1 < ref_start:
#             before_positions_possible.add(pos)

#         # Case: inside ALT region
#         elif ref_start <= pos < alt_end:
#             rel_pos = pos - ref_start
#             before_pos = ref_start + rel_pos
#             # only valid if inside original ref segment
#             if before_pos >= ref_start and before_pos < ref_end:
#                 before_positions_possible.add(before_pos)

#         # Case: after region, shift backwards
#         elif pos >= alt_end:
#             before_pos = pos - Δ
#             before_positions_possible.add(before_pos)

#     for pos in RG_after:
#         # A new RG is one whose mapped-before position wasn't an old RG position
#         mapped_candidates = []
#         # build same mapping as above quickly:
#         if pos + 1 < ref_start:
#             mapped_candidates = [pos]
#         elif ref_start <= pos < alt_end:
#             mapped_candidates = [ref_start + (pos - ref_start)]
#         else:
#             mapped_candidates = [pos - Δ]

#         if not any(c in RG_before for c in mapped_candidates):
#             new_RG.append(pos)

#     return {
#         "untouched_RG": sorted(list(set(untouched_RG))),
#         "lost_RG": sorted(list(set(lost_RG))),
#         "new_RG": sorted(list(set(new_RG))),
#     }


In [ ]:

# ###############################################
# # INPUT FILE PATHS
# ###############################################

# # Replace with your files:
# pos_bed = "/mnt/d/phd/scripts/14_gnomAD_conservation_proj/data/processed/pos_RGmotifs_coordinates.bed"
# neg_bed = "/mnt/d/phd/scripts/14_gnomAD_conservation_proj/data/processed/neg_RGmotifs_coordinates.bed"

# # pos_variants_file = "positive_variants.tsv"
# # neg_variants_file = "negative_variants.tsv"

# ###############################################
# # LOAD BED FILES
# ###############################################

# # BED columns: CHROM, START, END, STRAND, NAME, ...
# # We need CHROM, START, END, NAME
# bed_cols = ["CHROM", "START", "END", "STRAND", "NAME"]

# pos_regions = pd.read_csv(pos_bed, sep="\t", header=None, names=bed_cols+list(range(7)))
# neg_regions = pd.read_csv(neg_bed, sep="\t", header=None, names=bed_cols+list(range(7)))

# # Select only the important columns
# pos_regions = pos_regions[["CHROM", "START", "END", "STRAND", "NAME"]].copy()
# neg_regions = neg_regions[["CHROM", "START", "END", "STRAND", "NAME"]].copy()

# # Add group label
# pos_regions["group"] = "positive"
# neg_regions["group"] = "negative"

# # Combine region tables
# regions = pd.concat([pos_regions, neg_regions], ignore_index=True)

# ###############################################
# # PARSE REGION_ID FROM COLUMN 5
# ###############################################

# # Example NAME format:
# # Q12774_858_868_chr7_144365241_144365274_+

# def parse_region_id(name):
#     protein = name.split("_")[0]
#     rg_start = name.split("_")[1]
#     rg_end = name.split("_")[2]
#     region_id = f"{protein}_{rg_start}_{rg_end}"
#     return region_id

# regions["region_id"] = regions["NAME"].apply(parse_region_id)
# regions


# pr_regions = pr.PyRanges(regions.rename(columns={"CHROM":"Chromosome", "START": "Start", "END": "End"}))
# print(pr_regions)

In [ ]:
# with open('/mnt/d/phd/scripts/14_gnomAD_conservation_proj/data/processed/genomic_coordinates_info_pos.json', 'r') as f:
#     pos_genomic_coordinates_df = json.load(f)

# for i,el in enumerate(pos_genomic_coordinates_df):
#     curr_dict = pos_genomic_coordinates_df[i]
#     curr_dict["region_id"] = str(el['protein'] + '_' + str(el['prot_region'][0]) + '_' + str(el['prot_region'][1]))
#     pos_genomic_coordinates_df[i] = curr_dict
# with open('/mnt/d/phd/scripts/14_gnomAD_conservation_proj/data/processed/genomic_coordinates_info_neg.json', 'r') as f:
#     neg_genomic_coordinates_df = json.load(f)
# for i,el in enumerate(neg_genomic_coordinates_df):
#     curr_dict = neg_genomic_coordinates_df[i]
#     curr_dict["region_id"] = str(el['protein'] + '_' + str(el['prot_region'][0]) + '_' + str(el['prot_region'][1]))
#     neg_genomic_coordinates_df[i] = curr_dict

# merged_genomic_coordinates_list = [
#     {**d, "group": "pos"} for d in pos_genomic_coordinates_df
# ] + [
#     {**d, "group": "neg"} for d in neg_genomic_coordinates_df
# ]


# ###############################################
# # LOAD VARIANTS
# ###############################################

# # df_pos = pd.read_csv(pos_variants_file, sep="\t")
# df_pos = pd.read_csv("/mnt/d/phd/scripts/14_gnomAD_conservation_proj/data/output/combined_joint_variants_pos.tsv", sep="\t")
# # df_neg = pd.read_csv(neg_variants_file, sep="\t")
# df_neg = pd.read_csv("/mnt/d/phd/scripts/14_gnomAD_conservation_proj/data/output/combined_joint_variants_neg.tsv", sep="\t")

# df_pos["group"] = "positive"
# df_neg["group"] = "negative"

# variants = pd.concat([df_pos, df_neg], ignore_index=True)

# ###############################################
# # PREPARE PANDAS → PYRANGES OBJECTS
# ###############################################

# # Convert BED regions to pyranges
# pr_regions = pr.PyRanges(regions.rename(columns={"CHROM":"Chromosome", "START": "Start", "END": "End"}))
# print(pr_regions)
# # Convert variants into intervals (POS → POS+1)
# variants_interval = variants.rename(columns={"CHROM":"Chromosome", "POS":"Start"})
# variants_interval["End"] = variants_interval["Start"] + 1
# pr_variants = pr.PyRanges(variants_interval)

# ###############################################
# # OVERLAP: ASSIGN VARIANTS TO REGIONS
# ###############################################

# overlap = pr_variants.join(pr_regions)

# df_overlap = overlap.as_df()

# # Now df_overlap contains all matched variants with:
# # Chromosome, Start, End, REF, ALT, AF, region_id, group, etc.


In [ ]:
import json
import pandas as pd
import pyranges as pr
from collections import Counter


def load_json_regions_to_pyranges(json_file):
    """
    Load your protein-region JSON structure and convert it to a PyRanges object.
    Each JSON entry may contain multiple genomic intervals.
    Output columns:
        Chromosome, Start, End, Strand, protein, region_id, prot_region_start,
        prot_region_end, prot_seq, dna, group
    """
    with open(json_file, "r") as f:
        data = json.load(f)

    rows = []

    for entry in data:
        protein = entry.get("protein")
        region_id = entry.get("region_id")
        prot_region = entry.get("prot_region", [None, None])
        prot_seq = entry.get("prot_seq")
        dna = entry.get("dna")
        group = entry.get("group")

        for iv in entry["intervals"]:
            rows.append({
                "Chromosome": str(iv["chrom"]),
                "Start": int(iv["start"]),
                "End": int(iv["end"]),
                "Strand": iv.get("strand", "."),
                "protein": protein,
                "region_id": region_id,
                "prot_region_start": prot_region[0],
                "prot_region_end": prot_region[1],
                "prot_seq": prot_seq,
                "dna": dna,
                "group": group
            })

    df = pd.DataFrame(rows)

    return pr.PyRanges(df)

json_file = "/mnt/d/phd/scripts/14_gnomAD_conservation_proj/data/processed/genomic_coordinates_info_merged.json"

# # Load JSON
# with open(json_file, "r") as f:
#     data = json.load(f)
# # data = [{
# #         "protein": "Q9UPT8",
# #         "prot_region": [
# #             235,
# #             271
# #         ],
# #         "prot_seq": "SRGRGSRGRGRGYRGRGSRGGSRGRGMGRGSRGRGRG",
# #         "intervals": [
# #             {
# #                 "chrom": "19",
# #                 "start": 47089967,
# #                 "end": 47089979,
# #                 "strand": "-"
# #             },
# #             {
# #                 "chrom": "19",
# #                 "start": 47086441,
# #                 "end": 47086538,
# #                 "strand": "-"
# #             }
# #         ],
# #         "dna": "AGCCGCGGCCGAGGCAGCCGAGGCCGGGGCCGGGGCTACAGGGGCCGAGGAAGCCGTGGAGGATCGCGAGGCCGCGGCATGGGCAGGGGCAGCCGAGGCAGGGGCAGAGGC",
# #         "region_id": "Q9UPT8_235_271",
# #         "group": "pos"
# #     }]
# # Extract all chrom values from nested intervals
# chrom_values = []
# for entry in data:
#     for interval in entry.get("intervals", []):
#         chrom_values.append(interval.get("chrom"))

# # Count occurrences
# chrom_counts = Counter(chrom_values)

# # Print sorted output (largest first)
# print("Chromosome counts (sorted):")
# for chrom, count in chrom_counts.most_common():
#     print(f"{chrom}: {count}")

# # Optional DataFrame, sorted
# df_chrom = (
#     pd.DataFrame.from_dict(chrom_counts, orient="index", columns=["count"])
#       .sort_values("count", ascending=False)
# )
# print(df_chrom)


pr_regions = load_json_regions_to_pyranges(json_file)
# print(pr_regions)

###############################################
# LOAD VARIANTS
###############################################

df_pos = pd.read_csv("/mnt/d/phd/scripts/14_gnomAD_conservation_proj/data/output/combined_joint_variants_pos.tsv", sep="\t")
df_neg = pd.read_csv("/mnt/d/phd/scripts/14_gnomAD_conservation_proj/data/output/combined_joint_variants_neg.tsv", sep="\t")

df_pos["group"] = "positive"
df_neg["group"] = "negative"

variants = pd.concat([df_pos, df_neg], ignore_index=True)
# print(variants)
# print(variants["CHROM"].unique())
###############################################
# MAKE PYRANGES VARIANTS
###############################################

variants_interval = variants.rename(columns={"CHROM": "Chromosome", "POS": "Start"})
variants_interval["End"] = variants_interval["Start"] + 1
pr_variants = pr.PyRanges(variants_interval)
print(pr_variants)
# ###############################################
# OVERLAP USING PYRANGES
###############################################

overlap = pr_variants.join(pr_regions)
# print(overlap)
df_overlap = overlap.as_df()
print(df_overlap)